# Stage 2 — ResNet-50 & MobileNetV2 baselines on Food-101

Fine-tunes two ImageNet-pretrained baselines on Food-101 for comparison with SHViT.

**Setup:** `Runtime → Change runtime type → T4 GPU` before running.

Per model, the training script writes:
- `checkpoints/<model>/training_log.csv` — one row per epoch (lr, losses, top-1/5)
- `checkpoints/<model>/best.pth` — checkpoint of highest val top-1

> **Heads-up:** 50 epochs of ResNet-50 on Food-101 at batch 64 takes a few hours on a free-tier T4. Drop `--epochs` if you just want to smoke-test the pipeline.

## 0. Verify GPU

In [ ]:
import torch
print('PyTorch     :', torch.__version__)
print('CUDA avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))

## 1. (Optional) Mount Drive

Persist the dataset and checkpoints across Colab sessions.

In [ ]:
USE_DRIVE = True

# Every file this notebook saves lives under a single root directory called
# CV_Research_Paper_Food101 (on Drive if USE_DRIVE, else on local Colab disk).
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CV_Research_Paper_Food101'
else:
    BASE_DIR = '/content/CV_Research_Paper_Food101'

DATA_ROOT  = f'{BASE_DIR}/food101_data'
OUTPUT_DIR = f'{BASE_DIR}/baseline_checkpoints'

import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Base       :', BASE_DIR)
print('Dataset    :', DATA_ROOT)
print('Checkpoints:', OUTPUT_DIR)

In [ ]:
# Download Food-101 + the Tip-Adapter Zhou split into DATA_ROOT.
# torchvision lays images out as <DATA_ROOT>/food-101/images/<class>/*.jpg —
# exactly what tip_datasets.Food101 + split_zhou_Food101.json expect.
import os, torchvision

torchvision.datasets.Food101(root=DATA_ROOT, split='train', download=True)
torchvision.datasets.Food101(root=DATA_ROOT, split='test',  download=True)

SPLIT_PATH = f'{DATA_ROOT}/food-101/split_zhou_Food101.json'
if not os.path.exists(SPLIT_PATH):
    !pip install -q gdown
    !gdown "https://drive.google.com/uc?id=1QK0tGi096I0Ba6kggatX1ee6dJFIcEJl" -O "{SPLIT_PATH}"
print('Zhou split present:', os.path.exists(SPLIT_PATH))

# Optional speed-up: cache the dataset on local SSD (still under the
# CV_Research_Paper_Food101 root, just on Colab's local disk).
if USE_DRIVE:
    import shutil, time
    local = '/content/CV_Research_Paper_Food101/food101_data'
    if not os.path.exists(f'{local}/food-101/images'):
        print('Copying dataset to local SSD (~3-5 min, faster epochs)...')
        os.makedirs(os.path.dirname(local), exist_ok=True)
        t0 = time.time(); shutil.copytree(DATA_ROOT, local); print(f'done in {time.time()-t0:.0f}s')
    DATA_ROOT = local
    print('DATA_ROOT now:', DATA_ROOT)

## 2. Get the training script

Clone the project repo to pick up `train_baseline.py`.

In [ ]:
import os, shutil
REPO_DIR = '/content/Vision_Project_spring_26'
BRANCH   = 'Vision_Project_spring_26_Food101'
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git {REPO_DIR}

# Copy scripts + helper modules to /content for a clean working directory.
for fname in [
    'Stage 2: baseline models/train_baseline.py',
    'splits.py',
    'metrics.py',
    'augmentation.py',
]:
    shutil.copy(f'{REPO_DIR}/{fname}', f'/content/{os.path.basename(fname)}')
    print('Copied:', os.path.basename(fname))

# Copy the tip_datasets package (Tip-Adapter preprocessing + Zhou split loader).
if os.path.isdir('/content/tip_datasets'):
    shutil.rmtree('/content/tip_datasets')
shutil.copytree(f'{REPO_DIR}/tip_datasets', '/content/tip_datasets')
print('Copied: tip_datasets/')

## 3. Train ResNet-50

First run also downloads Food-101 (~5 GB, one-time).

In [ ]:
!python /content/train_baseline.py \
    --model resnet50 \
    --data-root {DATA_ROOT} \
    --output-dir {OUTPUT_DIR} \
    --epochs 50 \
    --batch-size 64 \
    --lr 1e-4 \
    --num-workers 8

## 4. Train MobileNetV2

In [ ]:
!python /content/train_baseline.py \
    --model mobilenet_v2 \
    --data-root {DATA_ROOT} \
    --output-dir {OUTPUT_DIR} \
    --epochs 50 \
    --batch-size 64 \
    --lr 1e-4 \
    --num-workers 8

## 5. Inspect logs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_name in ['resnet50', 'mobilenet_v2']:
    csv_path = f'{OUTPUT_DIR}/{model_name}/training_log.csv'
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    axes[0].plot(df['epoch'], df['train_loss'], label=f'{model_name} train')
    axes[0].plot(df['epoch'], df['val_loss'],   label=f'{model_name} val', linestyle='--')
    axes[1].plot(df['epoch'], df['val_top1'] * 100, label=f'{model_name} top-1')
    axes[1].plot(df['epoch'], df['val_top5'] * 100, label=f'{model_name} top-5', linestyle='--')
    print(f'{model_name}: best val top-1 = {df["val_top1"].max()*100:.2f}%')

axes[0].set_title('Loss');     axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()